# Treinando modelos

Normalmente os dados de bioacústica são desbalanceados. É comum que os animais que os ecologos e biológos tem interesse sejam animais em extinção.

A lib OpenSoundscape sugere tentar o balancear os dados ao máximo. A próxima célula faz upsample (repetindo amostras de dados) para igualar.

In [1]:
from pathlib import Path

import pandas as pd

from opensoundscape.data_selection import resample

/home/italo/open/shazam-da-natureza/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_file = Path("train.csv")
assert train_file.exists(), "Arquivo de treino não encontrado. Verifique que ele foi criado no '02_particionando_dados.ipynb'"

In [10]:
valid_file = Path("valid.csv")
assert valid_file.exists(), "Arquivo de validação não encontrado. Verifique que ele foi criado no '02_particionando_dados.ipynb'"

In [3]:
train_df = pd.read_csv(train_file, index_col=["file", "start_time", "end_time"])
train_df.head()

,,,NOCA,EATO,SCTA,BAWW,BCCH,AMCR,NOFL
file,start_time,end_time,,,,,,,
../data/mp3_Files/Recording_1/Recording_1_Segment_21.mp3,186.0,189.0,False,True,False,False,False,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_29.mp3,75.0,78.0,False,False,False,False,False,True,False
../data/mp3_Files/Recording_1/Recording_1_Segment_30.mp3,186.0,189.0,False,False,False,False,False,True,False
../data/mp3_Files/Recording_1/Recording_1_Segment_12.mp3,78.0,81.0,False,True,False,False,False,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_18.mp3,234.0,237.0,True,True,False,False,False,False,False


In [13]:
valid_df = pd.read_csv(train_file, index_col=["file", "start_time", "end_time"])
valid_df.head()

,,,NOCA,EATO,SCTA,BAWW,BCCH,AMCR,NOFL
file,start_time,end_time,,,,,,,
../data/mp3_Files/Recording_1/Recording_1_Segment_21.mp3,186.0,189.0,False,True,False,False,False,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_29.mp3,75.0,78.0,False,False,False,False,False,True,False
../data/mp3_Files/Recording_1/Recording_1_Segment_30.mp3,186.0,189.0,False,False,False,False,False,True,False
../data/mp3_Files/Recording_1/Recording_1_Segment_12.mp3,78.0,81.0,False,True,False,False,False,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_18.mp3,234.0,237.0,True,True,False,False,False,False,False


In [6]:
balanced_train_df = resample(train_df, n_samples_per_class=800, random_state=0)
balanced_train_df.head()

,,,NOCA,EATO,SCTA,BAWW,BCCH,AMCR,NOFL
file,start_time,end_time,,,,,,,
../data/mp3_Files/Recording_1/Recording_1_Segment_18.mp3,234.0,237.0,True,True,False,False,False,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_06.mp3,93.0,96.0,True,True,False,False,False,False,False
../data/mp3_Files/Recording_2/Recording_2_Segment_05.mp3,237.0,240.0,True,False,False,False,True,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_19.mp3,90.0,93.0,True,True,False,False,False,False,False
../data/mp3_Files/Recording_1/Recording_1_Segment_16.mp3,255.0,258.0,True,True,False,False,False,False,False


In [18]:
from opensoundscape import CNN

architecture = "resnet34"

class_list = list(train_df.columns)

model = CNN(
    architecture=architecture,
    classes=class_list,
    sample_duration=3,
    sample_rate=32000,
    device="cpu",
)

In [19]:
model.device

device(type='cpu')

In [8]:
checkpoint_folder = Path("model_training_checkpoints")
checkpoint_folder.mkdir(exist_ok=True)

In [ ]:
model.train(
    balanced_train_df,
    valid_df,
    steps=30,
    batch_size=64,
    log_interval=100,
    num_workers=1,
    wandb_session=None,
    save_path=checkpoint_folder,
)

  0%|                                                                                                                                                         | 0/30 [00:00<?, ?it/s]

Step: 0/30 (0.0%)
	Running Average Loss: 0.800
	Most Recent Step Loss: 0.800

Validation.



100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 72/72 [07:21<00:00,  6.13s/it]
/home/italo/open/shazam-da-natureza/venv/lib/python3.13/site-packages/opensoundscape/ml/cnn.py:1039: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  targets = torch.as_tensor(targets, dtype=torch.float32).to(self.device)
 27%|██████████████████████████████████████▋                                                                                                          | 8/30 [14:39<24:06, 65.76s/it]

In [ ]:
scores_df = model.predict(valid_df.head(), activation_layer="sigmoid")
scores_df.head()